In [19]:
import pandas as pd
from scipy.stats import kendalltau
import rbo

# -------- LOAD FILES --------
df_1 = pd.read_csv("./data/global_shap_model_1.csv")
df_2 = pd.read_csv("./data/global_shap_model_2.csv")
df_3 = pd.read_csv("./data/global_shap_model_3.csv")
df_4 = pd.read_csv("./data/global_shap_model_4.csv")
df_5 = pd.read_csv("./data/global_shap_model_5.csv")
df_6 = pd.read_csv("./data/global_shap_model_6.csv")

fusion_df = pd.read_csv("./data/fusion_results.csv")
stats_df = pd.read_csv("./data/p_values_for_base.csv")


# -------- HELPER FUNCTION: GET RANKING --------
def get_ranking(df, score_col, ascending=False):
    df_sorted = df.sort_values(by=score_col, ascending=ascending)
    return df_sorted["Feature"].tolist()


# -------- GET RANKINGS --------
r1 = get_ranking(df_1, "SHAP_Score")
r2 = get_ranking(df_2, "SHAP_Score")
r3 = get_ranking(df_3, "SHAP_Score")
r4 = get_ranking(df_4, "SHAP_Score")
r5 = get_ranking(df_5, "SHAP_Score")
r6 = get_ranking(df_6, "SHAP_Score")

fusion_rank = get_ranking(fusion_df, "Final_Global_Feature_Importance")

# lower p-value = more important
stats_rank = get_ranking(stats_df, "P-value", ascending=True)


# -------- METRIC FUNCTIONS --------
def kendall_tau_score(rank_a, rank_b):
    common = [f for f in rank_a if f in rank_b]

    rank_a_filtered = [f for f in rank_a if f in common]
    rank_b_filtered = [f for f in rank_b if f in common]

    pos_a = list(range(len(rank_a_filtered)))
    pos_b = [rank_b_filtered.index(f) for f in rank_a_filtered]

    tau, _ = kendalltau(pos_a, pos_b)
    return tau


def rbo_score(rank_a, rank_b, p=0.9):
    common = [f for f in rank_a if f in rank_b]

    rank_a_filtered = [f for f in rank_a if f in common]
    rank_b_filtered = [f for f in rank_b if f in common]

    return rbo.RankingSimilarity(rank_a_filtered, rank_b_filtered).rbo(p=p)


def recall_at_k(rank_a, rank_b, k):
    top_a = set(rank_a[:k])
    top_b = set(rank_b[:k])

    if len(top_a) == 0:
        return 0

    return len(top_a.intersection(top_b)) / len(top_a)


def compare_rankings(reference_rank, target_rank, reference_name, target_name):
    tau = kendall_tau_score(reference_rank, target_rank)
    rbo_val = rbo_score(reference_rank, target_rank)
    recall_10 = recall_at_k(reference_rank, target_rank, 10)
    recall_20 = recall_at_k(reference_rank, target_rank, 20)
    recall_30 = recall_at_k(reference_rank, target_rank, 30)

    return {
        "Reference": reference_name,
        "Model": target_name,
        "Kendall Tau": tau,
        "RBO": rbo_val,
        "Recall@10": recall_10,
        "Recall@20": recall_20,
        "Recall@30": recall_30
    }


# -------- STATS vs ALL SHAP MODELS + FUSION --------
results = []

results.append(compare_rankings(stats_rank, r1, "Stats", "SHAP1"))
results.append(compare_rankings(stats_rank, r2, "Stats", "SHAP2"))
results.append(compare_rankings(stats_rank, r3, "Stats", "SHAP3"))
results.append(compare_rankings(stats_rank, r4, "Stats", "SHAP4"))
results.append(compare_rankings(stats_rank, r5, "Stats", "SHAP5"))
results.append(compare_rankings(stats_rank, r6, "Stats", "SHAP6"))
results.append(compare_rankings(stats_rank, fusion_rank, "Stats", "Fusion"))

results_df = pd.DataFrame(results)

print(results_df)

  Reference   Model  Kendall Tau       RBO  Recall@10  Recall@20  Recall@30
0     Stats   SHAP1     0.120879  0.376579        0.2       0.30   0.466667
1     Stats   SHAP2     0.120213  0.369098        0.1       0.45   0.466667
2     Stats   SHAP3     0.682318  0.742883        0.7       0.85   0.966667
3     Stats   SHAP4    -0.219447  0.019393        0.0       0.00   0.066667
4     Stats   SHAP5     0.022977  0.063851        0.0       0.10   0.233333
5     Stats   SHAP6     0.023643  0.063851        0.0       0.10   0.233333
6     Stats  Fusion     0.413919  0.307886        0.1       0.35   0.633333


In [20]:
# -------- FORMAT TABLE (MATCH PDF STYLE) --------

# set Model as index
formatted_df = results_df.set_index("Model")

# transpose → rows become metrics, columns become models
formatted_df = formatted_df[[
    "Kendall Tau",
    "RBO",
    "Recall@10",
    "Recall@20",
    "Recall@30"
]].T

# optional: round values for nicer display
formatted_df = formatted_df.round(2)

# rename dataframe ML models
formatted_df.rename(columns={"SHAP1": "ALL: DT", 
                             "SHAP2": "ALL: SVM (rbf)",
                             "SHAP3": "ALL + AC: GNB",
                             "SHAP4": "AGG: Bagging, SVM (poly)",
                             "SHAP5": "AGG + AC: Boosting",
                             "SHAP6": "AGG + AC: GNB"}, inplace=True)

print(formatted_df)

Model        ALL: DT  ALL: SVM (rbf)  ALL + AC: GNB  AGG: Bagging, SVM (poly)  \
Kendall Tau     0.12            0.12           0.68                     -0.22   
RBO             0.38            0.37           0.74                      0.02   
Recall@10       0.20            0.10           0.70                      0.00   
Recall@20       0.30            0.45           0.85                      0.00   
Recall@30       0.47            0.47           0.97                      0.07   

Model        AGG + AC: Boosting  AGG + AC: GNB  Fusion  
Kendall Tau                0.02           0.02    0.41  
RBO                        0.06           0.06    0.31  
Recall@10                  0.00           0.00    0.10  
Recall@20                  0.10           0.10    0.35  
Recall@30                  0.23           0.23    0.63  


In [21]:
# find average STATS vs MEAN

# exclude Fusion column
ml_only_df = formatted_df.drop(columns=["Fusion"])

# compute mean across models (row-wise)
formatted_df["Mean of ML Methods"] = ml_only_df.mean(axis=1)

print(formatted_df)

Model        ALL: DT  ALL: SVM (rbf)  ALL + AC: GNB  AGG: Bagging, SVM (poly)  \
Kendall Tau     0.12            0.12           0.68                     -0.22   
RBO             0.38            0.37           0.74                      0.02   
Recall@10       0.20            0.10           0.70                      0.00   
Recall@20       0.30            0.45           0.85                      0.00   
Recall@30       0.47            0.47           0.97                      0.07   

Model        AGG + AC: Boosting  AGG + AC: GNB  Fusion  Mean of ML Methods  
Kendall Tau                0.02           0.02    0.41            0.123333  
RBO                        0.06           0.06    0.31            0.271667  
Recall@10                  0.00           0.00    0.10            0.166667  
Recall@20                  0.10           0.10    0.35            0.300000  
Recall@30                  0.23           0.23    0.63            0.406667  
